# Model tuning v002 — fixed whole-breath validation

Compare larger LightGBM models, ExtraTrees, and GRUs using the **same 20,000
training breaths and 6,791 validation breaths** as the accepted first baseline.
Every score is pooled MAE over inspiration (`u_out == 0`), in native pressure units.
No test labels or leaderboard scores select hyperparameters. Each trial records
settings, runtime, checkpoint, validation predictions, and per-R/C errors.

The notebook defaults to review mode. To run a new search, choose a new RUN_NAME
and enable RUN_TUNING, or execute with PA3_RUN_TUNING=1 in the project environment.
The executed research run used that environment flag; reopening without it reads
the saved results. Original baseline artifacts are retained.

In [1]:
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
import gc, hashlib, importlib.metadata as metadata, json, time
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
from IPython.display import display

RUN_TUNING = os.environ.get('PA3_RUN_TUNING', '0') == '1'
RESUME_TUNING = os.environ.get('PA3_RESUME_TUNING', '0') == '1'
RUN_NAME = 'tuning_v002'
SEED = 20261102
THREADS = 4
PROJECT_DIR = Path.cwd().resolve()
if not (PROJECT_DIR/'data').is_dir() and (PROJECT_DIR/'ventilator-pressure-prediction/data').is_dir():
    PROJECT_DIR /= 'ventilator-pressure-prediction'
DATA_DIR = PROJECT_DIR/'data'
OUT = PROJECT_DIR/'outputs'/RUN_NAME
BASE = PROJECT_DIR/'outputs/baseline_v001'
print('Mode:', 'execute new tuning search' if RUN_TUNING else 'review saved tuning results')
print('Results:', OUT)
if RUN_TUNING:
    import lightgbm as lgb
    from sklearn.ensemble import ExtraTreesRegressor
    import joblib

Mode: execute new tuning search
Results: /home/v2gao/dsc102/pa3/gym-workspace/ventilator-pressure-prediction/outputs/tuning_v002


## Data quality and preprocessing

The read-only audit covers schemas, missing/nonfinite values, legal input ranges,
IDs, 80-row breath structure, R/C consistency, timestamps, and exact duplicate
breaths. The source files are unchanged: no imputation, deletion, winsorization,
or clipping. Negative finite pressure measurements are retained.

Trees use physical feature values without standardization. GRUs fit input
mean/std and target median/IQR **only on the saved training breaths**, then invert
target scaling before scoring and prediction. Raw u_out masks remain separate.

In [2]:
quality = json.loads((PROJECT_DIR/'quality_audit_v002.json').read_text())
display(pd.DataFrame([
    ('Missing cells', quality['total_missing_cells']),
    ('Nonfinite cells', quality['total_nonfinite_cells']),
    ('Train/test breath overlap', quality['train_test_breath_id_overlap_count']),
    ('Duplicate training input breaths', quality['exact_input_breath_duplicates']['train.csv']['duplicate_breaths_beyond_first']),
    ('Exact test inputs matching training', quality['test_breaths_with_exact_input_match_in_train']),
    ('Rows removed', quality['cleaning_decision']['rows_removed']),
    ('Imputation needed', quality['cleaning_decision']['imputation_needed']),
], columns=['Check','Result']))

,Check,Result
0,Missing cells,0
1,Nonfinite cells,0
2,Train/test breath overlap,0
3,Duplicate training input breaths,0
4,Exact test inputs matching training,0
5,Rows removed,0
6,Imputation needed,False


## Shared feature builder — unchanged from baseline v001

In [3]:
STEPS = 80
DTYPES = {
    "id": "int64", "breath_id": "int32", "R": "int8", "C": "int8",
    "time_step": "float32", "u_in": "float32", "u_out": "int8",
    "pressure": "float32",
}


def digest(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def features(frame: pd.DataFrame) -> tuple[np.ndarray, list[str], np.ndarray]:
    """Return features in original row order, after verifying 80-row blocks."""
    if len(frame) % STEPS:
        raise ValueError("Unexpected breath length; do not reshape this file")
    blocks = frame["breath_id"].to_numpy().reshape(-1, STEPS)
    breath_ids = blocks[:, 0]
    if not np.all(blocks == breath_ids[:, None]):
        raise ValueError("Breaths are not contiguous 80-row blocks")
    if len(np.unique(breath_ids)) != len(breath_ids):
        raise ValueError("Repeated breath blocks")
    if not frame["id"].is_unique:
        raise ValueError("Duplicate row IDs")
    u = frame["u_in"].to_numpy().reshape(-1, STEPS)
    t = frame["time_step"].to_numpy().reshape(-1, STEPS)
    if not np.all(np.diff(t, axis=1) > 0):
        raise ValueError("Timestamps must increase inside each breath")
    for column in ("R", "C"):
        values = frame[column].to_numpy().reshape(-1, STEPS)
        if not np.all(values == values[:, :1]):
            raise ValueError(f"{column} changes within a breath")

    columns = {}
    for column in ("R", "C", "time_step", "u_in", "u_out"):
        columns[column] = frame[column].to_numpy(dtype=np.float32)
    step = np.arange(STEPS)
    dt = np.diff(t, axis=1, prepend=t[:, :1])
    cumulative = np.cumsum(u, axis=1, dtype=np.float32)
    columns["step"] = np.broadcast_to(step, u.shape).ravel()
    columns["dt"] = dt.ravel()
    columns["control_sum"] = cumulative.ravel()
    columns["control_mean"] = (cumulative / (step + 1)).ravel()
    # Valve opening is not measured flow: this is a history proxy, not volume.
    columns["control_integral"] = np.cumsum(u * dt, axis=1).ravel()
    for lag in (1, 2, 4, 8):
        shifted = np.empty_like(u)
        shifted[:, :lag] = u[:, :1]
        shifted[:, lag:] = u[:, :-lag]
        columns[f"u_in_lag_{lag}"] = shifted.ravel()
        columns[f"u_in_change_{lag}"] = (u - shifted).ravel()
    padded_sum = np.concatenate([np.zeros((len(u), 1), dtype=np.float32), cumulative], axis=1)
    for window in (4, 8):
        begin = np.maximum(0, step + 1 - window)
        columns[f"u_in_mean_{window}"] = (
            (padded_sum[:, step + 1] - padded_sum[:, begin]) / (step + 1 - begin)
        ).ravel()
    names = list(columns)
    matrix = np.column_stack(list(columns.values())).astype(np.float32, copy=False)
    if not np.isfinite(matrix).all() or "pressure" in names:
        raise ValueError("Invalid features")
    return matrix, names, breath_ids

## Load the exact saved split

All new models share the same training sample and validation set. No new split is
chosen to make a result look better. Preprocessing never uses validation targets.
Results are tuned holdout estimates; they are not an untouched final-test estimate.

In [4]:
if RUN_TUNING:
    OUT.mkdir(parents=True,exist_ok=RESUME_TUNING)
    train=pd.read_csv(DATA_DIR/'train.csv',dtype=DTYPES)
    assert digest(DATA_DIR/'train.csv') == quality['files']['train.csv']['sha256']
    x,names,breath_ids=features(train)
    old_split=np.load(BASE/'split_breath_ids.npz')
    index=pd.Index(breath_ids)
    train_idx=np.sort(index.get_indexer(old_split['trained']))
    val_idx=np.sort(index.get_indexer(old_split['validation']))
    assert (train_idx>=0).all() and (val_idx>=0).all()
    assert not np.intersect1d(train_idx,val_idx).size
    train_blocks=np.zeros(len(breath_ids),bool); train_blocks[train_idx]=True
    val_blocks=np.zeros(len(breath_ids),bool); val_blocks[val_idx]=True
    train_mask=np.repeat(train_blocks,STEPS)&(train.u_out.to_numpy()==0)
    val_mask=np.repeat(val_blocks,STEPS)&(train.u_out.to_numpy()==0)
    y=train.pressure.to_numpy()
    Xtr,Ytr=x[train_mask],y[train_mask]
    Xva,Yva=x[val_mask],y[val_mask]
    rc_val_rows=train.loc[val_mask,['R','C']].reset_index(drop=True)
    val_ids=train.loc[val_mask,'id'].to_numpy()
    shape=(len(breath_ids),STEPS,len(names))
    Xtr_seq=np.ascontiguousarray(x.reshape(shape)[train_idx])
    Xva_seq=np.ascontiguousarray(x.reshape(shape)[val_idx])
    Ytr_seq=np.ascontiguousarray(y.reshape(-1,STEPS)[train_idx])
    Yva_seq=np.ascontiguousarray(y.reshape(-1,STEPS)[val_idx])
    out_tr=np.ascontiguousarray(train.u_out.to_numpy().reshape(-1,STEPS)[train_idx])
    out_va=np.ascontiguousarray(train.u_out.to_numpy().reshape(-1,STEPS)[val_idx])
    rc_val_breaths=train[['R','C']].to_numpy()[::STEPS][val_idx]
    assert int(train_mask.sum())==607415 and int(val_mask.sum())==206098
    if RESUME_TUNING:
        prior_split=np.load(OUT/'split_breath_ids.npz')
        assert np.array_equal(np.sort(prior_split['trained']),np.sort(breath_ids[train_idx]))
        assert np.array_equal(np.sort(prior_split['validation']),np.sort(breath_ids[val_idx]))
        prior_design=json.loads((OUT/'design.json').read_text())
        assert prior_design['seed']==SEED and prior_design['features']==names
        assert prior_design['source_train_sha256']==digest(DATA_DIR/'train.csv')
    np.savez_compressed(OUT/'split_breath_ids.npz',trained=breath_ids[train_idx],validation=breath_ids[val_idx])
    np.save(OUT/'validation_ids.npy',val_ids)
    (OUT/'design.json').write_text(json.dumps(dict(seed=SEED,features=names,
        training_breaths=len(train_idx),validation_breaths=len(val_idx),
        training_scored_rows=int(train_mask.sum()),validation_scored_rows=int(val_mask.sum()),
        source_train_sha256=digest(DATA_DIR/'train.csv'),
        versions={p:metadata.version(p) for p in ['lightgbm','scikit-learn','torch','numpy','pandas']}),indent=2))
    previous_rows=pd.read_csv(OUT/'results.csv').to_dict('records') if RESUME_TUNING else None
    rows=[]
    old_metrics=json.loads((BASE/'metrics.json').read_text())
    rows.append(dict(trial='baseline_v001',family='LightGBM',
        validation_mae=old_metrics['validation_inspiratory_mae'],
        fit_seconds=old_metrics['fit_seconds'],best_iteration=300,
        model_file=str(BASE/'model.txt'),
        parameters=json.dumps(old_metrics['model_parameters']),status='complete',source='prior accepted run'))
    if previous_rows is not None:
        rows=previous_rows
    del x,train
    gc.collect()
    print('Prepared identical split:',Xtr.shape,Xva.shape,Xtr_seq.shape)

Prepared identical split: (607415, 20) (206098, 20) (20000, 80, 20)


## Incremental result logging and comparable metrics

In [5]:
def save_rows():
    pd.DataFrame(rows).to_csv(OUT/'results.csv',index=False)

def record_result(trial,family,params,pred,seconds,model_file,best_iteration=None):
    pred=np.asarray(pred,dtype=np.float64)
    assert pred.shape==Yva.shape and np.isfinite(pred).all()
    error=np.abs(pred-Yva)
    value=float(error.sum(dtype=np.float64)/len(error))
    np.save(OUT/f'{trial}_validation_predictions.npy',pred)
    slices=rc_val_rows.copy(); slices['absolute_error']=error
    slices.groupby(['R','C']).absolute_error.agg(['mean','count']).to_csv(OUT/f'{trial}_by_rc.csv')
    row=dict(trial=trial,family=family,validation_mae=value,fit_seconds=seconds,
        best_iteration=best_iteration,model_file=str(model_file),parameters=json.dumps(params),status='complete',source='tuning_v002')
    rows.append(row); save_rows()
    with (OUT/'progress.jsonl').open('a') as stream:
        stream.write(json.dumps(dict(event='trial_complete',**row))+'\n')
    print(json.dumps(row),flush=True)
    return row

def record_failure(trial,family,params,error):
    row=dict(trial=trial,family=family,parameters=json.dumps(params),status='failed',error=str(error))
    rows.append(row); save_rows()
    with (OUT/'progress.jsonl').open('a') as stream:
        stream.write(json.dumps(dict(event='trial_failed',**row))+'\n')
    print(json.dumps(row),flush=True)

if RUN_TUNING:
    save_rows()

## LightGBM: tree count and complexity

The first candidate changes only the tree cap. Later candidates raise leaves and
bin resolution while varying learning rate and minimum leaf support. All use L1
loss and early stopping on the same validation MAE. No scaling is applied.

In [6]:
LGB_CONFIGS=[
 dict(name='lgb_1000_63',n_estimators=1000,num_leaves=63,learning_rate=.07,min_child_samples=80,max_bin=127,reg_lambda=1.),
 dict(name='lgb_2000_127',n_estimators=2000,num_leaves=127,learning_rate=.05,min_child_samples=40,max_bin=255,reg_lambda=1.),
 dict(name='lgb_2500_255',n_estimators=2500,num_leaves=255,learning_rate=.04,min_child_samples=20,max_bin=255,reg_lambda=2.),
]
def run_lgb(config):
    cfg=dict(config); trial=cfg.pop('name')
    if any(row.get('trial')==trial and row.get('status')=='complete' for row in rows):
        print('Reuse completed',trial,flush=True); return
    print('START',trial,flush=True)
    try:
        model=lgb.LGBMRegressor(**cfg,objective='regression_l1',subsample=.9,subsample_freq=1,
            colsample_bytree=.9,n_jobs=THREADS,random_state=SEED,
            deterministic=True,force_col_wise=True,verbosity=-1)
        started=time.monotonic()
        model.fit(Xtr,Ytr,feature_name=names,eval_set=[(Xva,Yva)],eval_metric='l1',
            callbacks=[lgb.early_stopping(75,first_metric_only=True),lgb.log_evaluation(250)])
        seconds=time.monotonic()-started
        pred=model.predict(Xva,num_iteration=model.best_iteration_)
        checkpoint=OUT/f'{trial}.txt'
        model.booster_.save_model(str(checkpoint),num_iteration=model.best_iteration_)
        record_result(trial,'LightGBM',cfg,pred,seconds,checkpoint,model.best_iteration_)
        del model,pred; gc.collect()
    except Exception as error:
        record_failure(trial,'LightGBM',cfg,error)

### 1000 trees / 63 leaves

In [7]:
if RUN_TUNING:
    run_lgb(LGB_CONFIGS[0])

Reuse completed lgb_1000_63


### 2000 trees / 127 leaves

In [8]:
if RUN_TUNING:
    run_lgb(LGB_CONFIGS[1])

Reuse completed lgb_2000_127


### 2500 trees / 255 leaves

In [9]:
if RUN_TUNING:
    run_lgb(LGB_CONFIGS[2])

Reuse completed lgb_2500_255


## ExtraTrees: a different tree ensemble

Randomized trees use squared-error splits for feasible runtime, but selection is
still by the official inspiration MAE. Depth, leaf support, feature sampling, and
forest size are tuned. A node cap bounds model memory. No scaling is applied.

In [10]:
ET_CONFIGS=[
 dict(name='extra_150_depth18',n_estimators=150,max_depth=18,min_samples_leaf=10,max_features=1.0,max_leaf_nodes=30000),
 dict(name='extra_250_depth24',n_estimators=250,max_depth=24,min_samples_leaf=4,max_features=.8,max_leaf_nodes=50000),
]
def run_extra(config):
    cfg=dict(config); trial=cfg.pop('name')
    if any(row.get('trial')==trial and row.get('status')=='complete' for row in rows):
        print('Reuse completed',trial,flush=True); return
    print('START',trial,flush=True)
    try:
        model=ExtraTreesRegressor(**cfg,criterion='squared_error',n_jobs=THREADS,random_state=SEED)
        started=time.monotonic(); model.fit(Xtr,Ytr); seconds=time.monotonic()-started
        pred=model.predict(Xva)
        checkpoint=OUT/f'{trial}.joblib'; joblib.dump(model,checkpoint,compress=1)
        record_result(trial,'ExtraTrees',cfg,pred,seconds,checkpoint)
        del model,pred; gc.collect()
    except Exception as error:
        record_failure(trial,'ExtraTrees',cfg,error)

### ExtraTrees configuration 1

In [11]:
if RUN_TUNING:
    run_extra(ET_CONFIGS[0])

Reuse completed extra_150_depth18


### ExtraTrees configuration 2

In [12]:
if RUN_TUNING:
    run_extra(ET_CONFIGS[1])

Reuse completed extra_250_depth24


## GRU: learn the sequence history

Three hidden-size/layer/dropout/learning-rate combinations are tested. Inputs are
standardized using only training breaths. Targets use training-only median/IQR
scaling; all reported errors and predictions are converted back to pressure units.
L1 loss is masked to inspiration. Best checkpoints are restored. Each candidate
has a 12-epoch cap, patience 3, and a 600-second limit checked between epochs.

In [13]:
if RUN_TUNING:
    # Notebook-ready GRU helpers. No code below executes training until fit_gru_trial is called.
    # Set this before the first CUDA operation in a fresh notebook kernel.
    import os
    os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')

    import copy
    import json
    import random
    import time
    from pathlib import Path

    import numpy as np
    import torch
    from torch import nn
    from torch.utils.data import DataLoader, TensorDataset

    GRU_CONFIGS = [
        dict(name='gru64_1layer', hidden=64, layers=1, dropout=0.0,
             lr=1e-3, batch_size=256, max_epochs=12, patience=3, max_seconds=600),
        dict(name='gru96_2layer', hidden=96, layers=2, dropout=0.10,
             lr=1e-3, batch_size=256, max_epochs=12, patience=3, max_seconds=600),
        dict(name='gru128_2layer', hidden=128, layers=2, dropout=0.15,
             lr=7.5e-4, batch_size=256, max_epochs=12, patience=3, max_seconds=600),
    ]


    def seed_gru(seed=42):
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.set_num_threads(4)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        # Strict setting surfaces unsupported nondeterministic operations rather than hiding them.
        torch.use_deterministic_algorithms(True)


    class PressureGRU(nn.Module):
        def __init__(self, n_features, hidden=64, layers=1, dropout=0.0):
            super().__init__()
            self.gru = nn.GRU(n_features, hidden, num_layers=layers,
                              batch_first=True, dropout=dropout if layers > 1 else 0.0)
            self.head = nn.Sequential(nn.Linear(hidden, 32), nn.SiLU(), nn.Linear(32, 1))

        def forward(self, x):
            # No supplied hidden state: every batch/breath starts with zero state.
            sequence, _ = self.gru(x)
            return self.head(sequence).squeeze(-1)


    def inspiration_mae(y_true, y_pred, u_out):
        mask = np.asarray(u_out) == 0
        if not np.any(mask):
            raise ValueError('No inspiration points to score')
        errors = np.abs(np.asarray(y_pred, dtype=np.float64) -
                        np.asarray(y_true, dtype=np.float64))
        return float(errors[mask].sum(dtype=np.float64) / mask.sum())


    def per_rc_mae(y_true, y_pred, u_out, rc):
        """rc is [n_breaths, 2], from original unscaled R and C, for diagnostics only."""
        if rc is None:
            return []
        rc = np.asarray(rc)
        if rc.shape != (len(y_true), 2):
            raise ValueError('rc must have shape [n_breaths, 2]')
        result = []
        for r, c in sorted({tuple(row) for row in rc.tolist()}):
            rows = (rc[:, 0] == r) & (rc[:, 1] == c)
            mask = np.asarray(u_out)[rows] == 0
            result.append(dict(R=float(r), C=float(c), breaths=int(rows.sum()),
                               scored_points=int(mask.sum()),
                               mae=inspiration_mae(np.asarray(y_true)[rows],
                                                   np.asarray(y_pred)[rows],
                                                   np.asarray(u_out)[rows])))
        return result


    @torch.no_grad()
    def _predict_gru_scaled(model, x_standardized, batch_size=512):
        model.eval()
        device = next(model.parameters()).device
        output = np.empty(x_standardized.shape[:2], dtype=np.float32)
        for start in range(0, len(x_standardized), batch_size):
            stop = min(start + batch_size, len(x_standardized))
            batch = torch.from_numpy(np.ascontiguousarray(x_standardized[start:stop])).to(device)
            output[start:stop] = model(batch).cpu().numpy()
        return output


    def predict_gru(bundle, x, batch_size=512):
        """Returns all timesteps in original array order, in native pressure units."""
        x = np.asarray(x, dtype=np.float32)
        if x.ndim != 3 or x.shape[1:] != bundle['input_shape']:
            raise ValueError('Unexpected sequence/feature shape')
        if not np.isfinite(x).all():
            raise ValueError('Nonfinite GRU input')
        standardized = np.ascontiguousarray((x - bundle['x_mean']) / bundle['x_std'])
        scaled = _predict_gru_scaled(bundle['model'], standardized, batch_size)
        output = scaled * bundle['target_scale'] + bundle['target_center']
        if not np.isfinite(output).all():
            raise ValueError('Nonfinite GRU prediction')
        return output.astype(np.float32, copy=False)


    def fit_gru_trial(x_train, y_train, u_out_train, x_val, y_val, u_out_val,
                      config, seed=42, val_rc=None, checkpoint_path=None,
                      epoch_jsonl_path=None, on_epoch=None):
        """Input arrays: X=[breaths,80,features], y/u_out=[breaths,80].

        Supply ONLY the selected training breaths in x_train/y_train. Validation must
        contain disjoint whole breaths. No target pressure is included among X features.
        Scaling is fitted exclusively here on training inputs/targets; val_rc never fits.
        max_seconds is checked between epochs, so a trial may exceed it by one epoch.
        Returns (bundle, report, full_validation_predictions).
        """
        if not torch.cuda.is_available():
            raise RuntimeError('A CUDA GPU is required for these GRU trials')
        seed_gru(seed)
        config = dict(config)
        x_train = np.asarray(x_train, dtype=np.float32)
        x_val = np.asarray(x_val, dtype=np.float32)
        y_train = np.asarray(y_train, dtype=np.float32)
        y_val = np.asarray(y_val, dtype=np.float32)
        u_out_train = np.asarray(u_out_train)
        u_out_val = np.asarray(u_out_val)
        if x_train.ndim != 3 or x_train.shape[1] != 80 or x_val.shape[1:] != x_train.shape[1:]:
            raise ValueError('Expected compatible 80-timestep sequence arrays')
        for x, y, out in ((x_train, y_train, u_out_train), (x_val, y_val, u_out_val)):
            if y.shape != x.shape[:2] or out.shape != y.shape:
                raise ValueError('Input/target/mask shapes disagree')
            if not np.isfinite(x).all() or not np.isfinite(y).all():
                raise ValueError('Nonfinite training/validation values')
            if not np.isin(out, [0, 1]).all() or not np.any(out == 0):
                raise ValueError('Invalid inspiration mask')

        x_mean = x_train.mean(axis=(0, 1), dtype=np.float64).astype(np.float32)
        x_std = x_train.std(axis=(0, 1), dtype=np.float64).astype(np.float32)
        x_std[x_std < 1e-6] = 1.0
        train_z = np.ascontiguousarray((x_train - x_mean) / x_std)
        val_z = np.ascontiguousarray((x_val - x_mean) / x_std)
        train_mask = (u_out_train == 0).astype(np.float32)
        scored_targets = y_train[u_out_train == 0]
        center = float(np.median(scored_targets))
        q25, q75 = np.quantile(scored_targets, [0.25, 0.75])
        scale = max(float(q75 - q25), 1.0)
        train_y_z = np.ascontiguousarray((y_train - center) / scale)
        dataset = TensorDataset(torch.from_numpy(train_z), torch.from_numpy(train_y_z),
                                torch.from_numpy(train_mask))
        generator = torch.Generator().manual_seed(seed)
        loader = DataLoader(dataset, batch_size=config['batch_size'], shuffle=True,
                            num_workers=0, pin_memory=True, generator=generator)
        model = PressureGRU(x_train.shape[-1], config['hidden'], config['layers'],
                            config['dropout']).cuda()
        optimizer = torch.optim.Adam(model.parameters(), lr=config['lr'])
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min',
                                                             factor=0.5, patience=1)
        best_mae, best_epoch, best_state = float('inf'), 0, None
        history, bad_epochs = [], 0
        started = time.monotonic()
        torch.cuda.reset_peak_memory_stats()
        for epoch in range(1, config['max_epochs'] + 1):
            epoch_started = time.monotonic()
            model.train()
            train_error, train_count = 0.0, 0.0
            for xb, yb, mask in loader:
                xb = xb.cuda(non_blocking=True)
                yb = yb.cuda(non_blocking=True)
                mask = mask.cuda(non_blocking=True)
                optimizer.zero_grad(set_to_none=True)
                pred = model(xb)
                absolute = (pred - yb).abs() * mask
                count = mask.sum()
                if count.item() == 0:
                    continue
                loss = absolute.sum() / count
                if not torch.isfinite(loss):
                    raise RuntimeError('Nonfinite GRU loss')
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                train_error += float(absolute.detach().sum()) * scale
                train_count += float(count)
            val_predictions = _predict_gru_scaled(model, val_z) * scale + center
            val_mae = inspiration_mae(y_val, val_predictions, u_out_val)
            if val_mae < best_mae - 1e-6:
                best_mae, best_epoch, bad_epochs = val_mae, epoch, 0
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                if checkpoint_path is not None:
                    path = Path(checkpoint_path)
                    path.parent.mkdir(parents=True, exist_ok=True)
                    temporary = path.with_name(path.name + '.tmp')
                    torch.save(dict(state_dict=best_state, config=config, seed=seed,
                                    n_features=x_train.shape[-1], x_mean=x_mean, x_std=x_std,
                                    target_center=center, target_scale=scale,
                                    best_epoch=best_epoch, best_val_mae=best_mae), temporary)
                    os.replace(temporary, path)
            else:
                bad_epochs += 1
            scheduler.step(val_mae)
            record = dict(trial=config['name'], seed=seed, epoch=epoch,
                          train_inspiratory_mae=train_error / train_count,
                          validation_inspiratory_mae=val_mae, best_val_mae=best_mae,
                          best_epoch=best_epoch, lr=optimizer.param_groups[0]['lr'],
                          epoch_seconds=time.monotonic() - epoch_started,
                          elapsed_seconds=time.monotonic() - started,
                          peak_gpu_memory_MiB=torch.cuda.max_memory_allocated() / 2**20)
            history.append(record)
            if epoch_jsonl_path is not None:
                path = Path(epoch_jsonl_path)
                path.parent.mkdir(parents=True, exist_ok=True)
                with path.open('a') as stream:
                    stream.write(json.dumps(record) + '\n')
            if on_epoch is not None:
                on_epoch(record)
            if bad_epochs >= config['patience'] or record['elapsed_seconds'] >= config['max_seconds']:
                break
        if best_state is None:
            raise RuntimeError('No valid GRU checkpoint produced')
        model.load_state_dict(best_state)
        model.eval()
        bundle = dict(model=model, config=config, seed=seed, x_mean=x_mean, x_std=x_std,
                      target_center=center, target_scale=scale, input_shape=x_train.shape[1:])
        val_predictions = predict_gru(bundle, x_val)
        restored_mae = inspiration_mae(y_val, val_predictions, u_out_val)
        if not np.isclose(restored_mae, best_mae, rtol=1e-6, atol=1e-6):
            raise RuntimeError('Restored checkpoint MAE differs from logged best')
        report = dict(config=config, seed=seed, best_epoch=best_epoch,
                      validation_inspiratory_mae=restored_mae,
                      elapsed_seconds=time.monotonic() - started,
                      train_breaths=len(x_train), val_breaths=len(x_val),
                      scored_val_points=int((u_out_val == 0).sum()),
                      per_rc=per_rc_mae(y_val, val_predictions, u_out_val, val_rc),
                      epochs=history)
        return bundle, report, val_predictions


    def predictions_by_rc(predictions, rc):
        """Optional raw slices; keep these arrays outside JSON reports."""
        rc = np.asarray(rc)
        predictions = np.asarray(predictions)
        if rc.shape != (len(predictions), 2):
            raise ValueError('rc must have shape [n_breaths,2]')
        result = {}
        for r, c in sorted({tuple(row) for row in rc.tolist()}):
            rows = np.flatnonzero((rc[:, 0] == r) & (rc[:, 1] == c))
            result[(r, c)] = dict(breath_rows=rows, predictions=predictions[rows])
        return result

In [14]:
def run_gru(config):
    trial=config['name']
    if any(row.get('trial')==trial and row.get('status')=='complete' for row in rows):
        print('Reuse completed',trial,flush=True); return
    print('START',trial,flush=True)
    try:
        bundle,report,pred=fit_gru_trial(
            Xtr_seq,Ytr_seq,out_tr,Xva_seq,Yva_seq,out_va,
            config,seed=SEED,val_rc=rc_val_breaths,
            checkpoint_path=OUT/f'{trial}.pt',epoch_jsonl_path=OUT/'epochs.jsonl',
            on_epoch=lambda row: print(json.dumps(row),flush=True))
        (OUT/f'{trial}_report.json').write_text(json.dumps(report,indent=2))
        record_result(trial,'GRU',config,pred[out_va==0],report['elapsed_seconds'],
            OUT/f'{trial}.pt',report['best_epoch'])
        del bundle,pred; gc.collect(); torch.cuda.empty_cache()
    except Exception as error:
        record_failure(trial,'GRU',config,error)
        gc.collect()
        if 'torch' in globals(): torch.cuda.empty_cache()

### GRU configuration 1

In [15]:
if RUN_TUNING:
    run_gru(GRU_CONFIGS[0])

Reuse completed gru64_1layer


### GRU configuration 2

In [16]:
if RUN_TUNING:
    run_gru(GRU_CONFIGS[1])

Reuse completed gru96_2layer


### GRU configuration 3

In [17]:
if RUN_TUNING:
    run_gru(GRU_CONFIGS[2])

Reuse completed gru128_2layer


## Longer GRU follow-ups

The initial GRU runs all reached the 12-epoch cap while still improving. Two
follow-ups isolate a longer training budget: the same 96/128-unit architectures
and learning rates, now up to 100 epochs with patience 8. Initial trial results
and checkpoints remain intact. Resume mode reuses completed trials and verifies
that the saved split, seed, features, and data fingerprint match.

In [18]:
if RUN_TUNING:
    GRU_LONG_CONFIGS=[
        dict(GRU_CONFIGS[1],name='gru96_2layer_100epochs',max_epochs=100,patience=8),
        dict(GRU_CONFIGS[2],name='gru128_2layer_100epochs',max_epochs=100,patience=8),
    ]

In [19]:
if RUN_TUNING:
    run_gru(GRU_LONG_CONFIGS[0])

START gru96_2layer_100epochs
{"trial": "gru96_2layer_100epochs", "seed": 20261102, "epoch": 1, "train_inspiratory_mae": 3.9311085062560363, "validation_inspiratory_mae": 2.4679714715395007, "best_val_mae": 2.4679714715395007, "best_epoch": 1, "lr": 0.001, "epoch_seconds": 1.5282435999251902, "elapsed_seconds": 1.5283206361345947, "peak_gpu_memory_MiB": 361.06298828125}
{"trial": "gru96_2layer_100epochs", "seed": 20261102, "epoch": 2, "train_inspiratory_mae": 2.058860551755063, "validation_inspiratory_mae": 2.043017314837599, "best_val_mae": 2.043017314837599, "best_epoch": 2, "lr": 0.001, "epoch_seconds": 1.2107298523187637, "elapsed_seconds": 2.740582540165633, "peak_gpu_memory_MiB": 361.06298828125}
{"trial": "gru96_2layer_100epochs", "seed": 20261102, "epoch": 3, "train_inspiratory_mae": 1.7735587729410056, "validation_inspiratory_mae": 1.667909445810496, "best_val_mae": 1.667909445810496, "best_epoch": 3, "lr": 0.001, "epoch_seconds": 1.2122457078658044, "elapsed_seconds": 3.954042

In [20]:
if RUN_TUNING:
    run_gru(GRU_LONG_CONFIGS[1])

START gru128_2layer_100epochs
{"trial": "gru128_2layer_100epochs", "seed": 20261102, "epoch": 1, "train_inspiratory_mae": 3.9529478860232157, "validation_inspiratory_mae": 2.3739636776963833, "best_val_mae": 2.3739636776963833, "best_epoch": 1, "lr": 0.00075, "epoch_seconds": 1.3515780279412866, "elapsed_seconds": 1.3516504568979144, "peak_gpu_memory_MiB": 453.2568359375}
{"trial": "gru128_2layer_100epochs", "seed": 20261102, "epoch": 2, "train_inspiratory_mae": 2.069672091884147, "validation_inspiratory_mae": 1.8670153809947387, "best_val_mae": 1.8670153809947387, "best_epoch": 2, "lr": 0.00075, "epoch_seconds": 1.3515323707833886, "elapsed_seconds": 2.7044059187173843, "peak_gpu_memory_MiB": 453.2568359375}
{"trial": "gru128_2layer_100epochs", "seed": 20261102, "epoch": 3, "train_inspiratory_mae": 1.7307294814073801, "validation_inspiratory_mae": 1.7980807556005483, "best_val_mae": 1.7980807556005483, "best_epoch": 3, "lr": 0.00075, "epoch_seconds": 1.3557893270626664, "elapsed_secon

## Results and selected test predictions

Compare all completed configurations on the identical holdout. Only the selected
candidate generates a new test CSV. This notebook does not automatically upload
each trial; the selected CSV can be scored once after review of the search.

In [21]:
results=pd.read_csv(OUT/'results.csv')
display(results.sort_values('validation_mae',na_position='last')[
    ['trial','family','validation_mae','fit_seconds','best_iteration','status','parameters']])
if RUN_TUNING:
    complete=results[results.status=='complete'].sort_values('validation_mae')
    best=complete.iloc[0].to_dict()
    print('Selected:',best['trial'],best['validation_mae'])
    test=pd.read_csv(DATA_DIR/'test.csv',dtype=DTYPES)
    Xtest,test_names,test_breaths=features(test)
    assert test_names==names and not np.intersect1d(breath_ids,test_breaths).size
    if best['family']=='LightGBM':
        chosen=lgb.Booster(model_file=best['model_file'])
        prediction=chosen.predict(Xtest,num_threads=THREADS)
    elif best['family']=='ExtraTrees':
        chosen=joblib.load(best['model_file']); prediction=chosen.predict(Xtest)
    else:
        saved=torch.load(best['model_file'],map_location='cpu')
        cfg=saved['config']
        chosen=PressureGRU(saved['n_features'],cfg['hidden'],cfg['layers'],cfg['dropout']).cuda()
        chosen.load_state_dict(saved['state_dict'])
        bundle=dict(model=chosen,x_mean=saved['x_mean'],x_std=saved['x_std'],
            target_center=saved['target_center'],target_scale=saved['target_scale'],input_shape=(80,len(names)))
        prediction=predict_gru(bundle,Xtest.reshape(-1,80,len(names))).ravel()
    sample=pd.read_csv(DATA_DIR/'sample_submission.csv')
    ordered=pd.Series(prediction,index=test.id.to_numpy()).reindex(sample.id).to_numpy()
    assert len(ordered)==603600 and np.isfinite(ordered).all()
    submission=pd.DataFrame({'id':sample.id.to_numpy(),'pressure':ordered})
    submission.to_csv(OUT/'submission.csv',index=False,float_format='%.8f')
    check=pd.read_csv(OUT/'submission.csv')
    assert list(check.columns)==['id','pressure'] and np.array_equal(check.id.to_numpy(),sample.id.to_numpy())
    assert np.isfinite(check.pressure).all()
    best.update(submission_rows=len(check),submission_sha256=digest(OUT/'submission.csv'),
        original_baseline_validation_mae=1.0108896728514998,
        validation_improvement_percent=(1-best['validation_mae']/1.0108896728514998)*100)
    (OUT/'selected_model.json').write_text(json.dumps(best,indent=2))
    print('Selected CSV verified:',best)
else:
    if (OUT/'selected_model.json').exists():
        display(json.loads((OUT/'selected_model.json').read_text()))
    if (OUT/'submission_receipt.json').exists():
        display(json.loads((OUT/'submission_receipt.json').read_text()))

Selected: lgb_2500_255 0.7180270039640434
Selected CSV verified: {'trial': 'lgb_2500_255', 'family': 'LightGBM', 'validation_mae': 0.7180270039640434, 'fit_seconds': 94.99919431796296, 'best_iteration': 2500.0, 'model_file': '/home/v2gao/dsc102/pa3/gym-workspace/ventilator-pressure-prediction/outputs/tuning_v002/lgb_2500_255.txt', 'parameters': '{"n_estimators": 2500, "num_leaves": 255, "learning_rate": 0.04, "min_child_samples": 20, "max_bin": 255, "reg_lambda": 2.0}', 'status': 'complete', 'source': 'tuning_v002', 'submission_rows': 603600, 'submission_sha256': '1f8a028bf2bd144fa74f4eb843bb20783de2b86fd8927ec85108d96d8e954d62', 'original_baseline_validation_mae': 1.0108896728514998, 'validation_improvement_percent': 28.97078452303845}


,trial,family,validation_mae,fit_seconds,best_iteration,status,parameters
3,lgb_2500_255,LightGBM,0.718027,94.999194,2500.0,complete,"{""n_estimators"": 2500, ""num_leaves"": 255, ""lea..."
5,extra_250_depth24,ExtraTrees,0.738020,121.911908,NaN,complete,"{""n_estimators"": 250, ""max_depth"": 24, ""min_sa..."
2,lgb_2000_127,LightGBM,0.762585,64.243209,2000.0,complete,"{""n_estimators"": 2000, ""num_leaves"": 127, ""lea..."
10,gru128_2layer_100epochs,GRU,0.837811,122.251083,81.0,complete,"{""name"": ""gru128_2layer_100epochs"", ""hidden"": ..."
9,gru96_2layer_100epochs,GRU,0.857128,75.945264,54.0,complete,"{""name"": ""gru96_2layer_100epochs"", ""hidden"": 9..."
1,lgb_1000_63,LightGBM,0.866746,30.320141,1000.0,complete,"{""n_estimators"": 1000, ""num_leaves"": 63, ""lear..."
4,extra_150_depth18,ExtraTrees,0.882724,72.649132,NaN,complete,"{""n_estimators"": 150, ""max_depth"": 18, ""min_sa..."
0,baseline_v001,LightGBM,1.010890,10.342663,300.0,complete,"{""boosting_type"": ""gbdt"", ""class_weight"": null..."
7,gru96_2layer,GRU,1.133280,14.730907,12.0,complete,"{""name"": ""gru96_2layer"", ""hidden"": 96, ""layers..."
8,gru128_2layer,GRU,1.159091,16.425593,12.0,complete,"{""name"": ""gru128_2layer"", ""hidden"": 128, ""laye..."
